# CS781 Phase 4 - Court Judgment Prediction (CJPE) on top of the Phase-1 models

One notebook for every Phase-4 experiment, built like the Phase-2 and Phase-3 notebooks. Set the experiment in **section 3** and run all, once per experiment.

| switch | options |
|---|---|
| `MODEL_TYPE` | `"gpt"`, `"modern"`, `"rnn"`, `"t5"` (your Phase-1 models; for `t5` only the encoder is used) |
| `FT_METHOD` | `"full"`, `"partial"` (top layers only), `"bitfit"` (bias terms only), `"lorma"`, `"adapter"` (bottleneck adapters), `"head_only"` |
| `USE_KD` | `True`: train a smaller student (first `STUDENT_LAYERS` layers of `MODEL_TYPE`) from a saved teacher |
| `SECTION` | which part of the judgment is read: `"last"`, `"first"`, `"first_last"`, `"spread"` (chunks spread over the whole document) |
| `N_CHUNKS` | number of 512-token chunks per document (1 = a single window, >1 = hierarchical) |
| `DOC_POOL` | how chunk vectors become one document vector: `"mean"`, `"attention"` (learned weights), `"bigru"` |

Pipeline: judgment -> Phase-1 BPE tokens -> `N_CHUNKS` chunks of 512 tokens chosen by `SECTION` -> Phase-1 model -> one vector per chunk (mean of its token states) -> document vector (`DOC_POOL`) -> linear layer -> 2 classes -> threshold tuned on dev for macro-F1 -> `prediction.json`.

## 1. Hugging Face Login

In [ ]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("hf_token")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

## 2. Check GPU

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
  print(f"GPU {i}:", torch.cuda.get_device_name(i))

## 3. Configuration

In [ ]:
import os
import random
import json
import time
import math
import zipfile
import gc
import torch

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TRAIN_FILE = "Phase_4_CJPE/Train/train.json"
DEV_FILE = "Phase_4_CJPE/Train/dev.json"
TEST_FILE = "Phase_4_CJPE/Test/test.json"

# ---------------- the experiment ----------------
EXPERIMENT_NAME = "modern_lorma_last2_mean"    # <-- CHANGED for this run
MODEL_TYPE = "modern"    # <-- CHANGED for this run
FT_METHOD = "lorma"    # <-- CHANGED for this run
SMOKE_TEST = False
USE_PHASE1_WEIGHTS = True    # False = random initialisation (ablation only)

# long documents
CHUNK_TOKENS = 512           # tokens per chunk
N_CHUNKS = 2                 # chunks per document (1 = one window; >1 = hierarchical chunk -> document model)
SECTION = "last"             # "last" | "first" | "first_last" | "spread" (N_CHUNKS chunks spread evenly over the document)
DOC_POOL = "mean"    # <-- CHANGED for this run
DOC_GRU_HIDDEN = 256
CLASS_WEIGHTING = "inverse"  # "none" | "inverse": loss weight of a class = N / (2 * count)
THRESHOLD_TUNING = True      # choose the decision threshold on P(label 1) that maximises dev macro-F1
MAX_TRAIN_DOCS = 8000        # random subset of the training documents (None = all); keeps one run under ~1 hour

# knowledge distillation (Section 10.2.2): student = first STUDENT_LAYERS layers of MODEL_TYPE
USE_KD = False
STUDENT_LAYERS = 2
TEACHER_MODEL_TYPE = "modern"
TEACHER_FT_METHOD = "full"
TEACHER_CJPE_PATH = "/kaggle/input/datasets/medinim/phase4-teacher/cjpe_model.pt"
KD_TEMPERATURE = 2.0
KD_ALPHA = 0.5               # loss = KD_ALPHA * CE(gold) + (1 - KD_ALPHA) * T^2 * KL(teacher || student)
# (the teacher must have been trained with the same CHUNK_TOKENS, N_CHUNKS, SECTION and DOC_POOL)

# LoRMA / adapters / partial
ADAPTER_RANK = 8
ADAPTER_ALPHA = 16
BOTTLENECK = 64              # "adapter": size of the bottleneck layer
PARTIAL_TOP_LAYERS = 2       # "partial": only the top layers (+ final norm + head) are trained

# Phase-1 models: one Kaggle dataset with model_<type>.safetensors and bpe_legal_<type>.json
PHASE1_FOLDER = "/kaggle/input/datasets/medinim/phase1-models"   # searched automatically below if this path is wrong
if not os.path.exists(os.path.join(PHASE1_FOLDER, "model_modern.safetensors")):
  import glob
  hits = glob.glob("/kaggle/input/**/model_modern.safetensors", recursive=True)
  if hits:
    PHASE1_FOLDER = os.path.dirname(hits[0])
print("Phase-1 models from:", PHASE1_FOLDER)
PHASE1_MODEL_FILES = {k: os.path.join(PHASE1_FOLDER, f"model_{k}.safetensors") for k in ["gpt", "modern", "rnn", "t5"]}
PHASE1_BPE_FILES = {k: os.path.join(PHASE1_FOLDER, f"bpe_legal_{k}.json") for k in ["gpt", "modern", "rnn", "t5"]}

# Phase-1 architecture sizes: must be EXACTLY what Phase 1 trained with
MODEL_CONTEXT_LENGTH = 2048
N_EMBD, N_LAYER, N_HEAD, N_KV_HEAD = 640, 10, 10, 5
D_FF = 4 * N_EMBD
RNN_CELL, RNN_HIDDEN, RNN_LAYERS = "lstm", 1152, 4
T5_D_MODEL, T5_ENC_LAYERS, T5_DEC_LAYERS, T5_HEADS, T5_D_FF, T5_N_SENTINELS = 512, 8, 8, 8, 2048, 100
DROPOUT = 0.1
LOAD_CJPE_PATH = None        # path to a saved cjpe_model.pt -> skip training, only predict
REQUIRE_GPU = True

# training (Trainer, like the earlier phases)
if FT_METHOD in ("full", "partial") or USE_KD:
  LEARNING_RATE = 5e-5
elif FT_METHOD == "bitfit":
  LEARNING_RATE = 1e-3
else:
  LEARNING_RATE = 5e-4       # LoRMA / adapters / a fresh head
EPOCHS = 3
BATCH_SIZE = max(1, 8 // N_CHUNKS)     # documents per GPU; every document is N_CHUNKS sequences of 512 tokens
GRAD_ACCUM = max(1, 8 // BATCH_SIZE)   # 2 GPUs x BATCH_SIZE x GRAD_ACCUM = 16 documents per update
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

if SMOKE_TEST:
  EPOCHS = 1
  MAX_TRAIN_DOCS = 64
if REQUIRE_GPU and not torch.cuda.is_available():
  raise RuntimeError("No GPU! In Kaggle: Settings -> Accelerator -> GPU T4 x2, then Save & Run All again.")
if USE_KD and not os.path.exists(TEACHER_CJPE_PATH):
  import glob
  hits = glob.glob("/kaggle/input/**/cjpe_model.pt", recursive=True)
  if hits:
    TEACHER_CJPE_PATH = hits[0]
  print("KD teacher:", TEACHER_CJPE_PATH)

OUTPUT_DIR = "/kaggle/working/cs781_phase4"
EXP_DIR = os.path.join(OUTPUT_DIR, EXPERIMENT_NAME)
os.makedirs(EXP_DIR, exist_ok=True)
PREDICTION_PATH = os.path.join(EXP_DIR, "prediction.json")
SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, f"submission_{EXPERIMENT_NAME}.zip")
RESULTS_PATH = os.path.join(OUTPUT_DIR, "results.jsonl")
print("Experiment:", EXPERIMENT_NAME, "|", MODEL_TYPE, FT_METHOD, f"| {N_CHUNKS} x {CHUNK_TOKENS} tokens, {SECTION}, {DOC_POOL}", "| KD" if USE_KD else "")

## 4. Tokenizer (the Phase-1 BPE, unchanged)

In [ ]:
import json
import heapq
from collections import Counter, defaultdict

SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]
WORD_END = "</w>"

class ByteLevelBPE:
  def __init__(self, vocab_size):
      self.vocab_size = vocab_size
      self.merges = []
      self.merge_ranks = {}
      self.itos = []
      self.stoi = {}
      self._cache = {}
      self.pad_id = None
      self.bos_id = None
      self.eos_id = None

  def _word_to_symbols(self, word):
      byte_vals = word.encode("utf-8")
      symbols = [chr(b) for b in byte_vals]
      symbols.append(WORD_END)
      return symbols

  def train(self, sentences):
      word_freq = Counter()
      for sent in sentences:
          for word in sent.split():
              word_freq[word] += 1

      words = list(word_freq)
      freqs = [word_freq[w] for w in words]
      word_symbols = [self._word_to_symbols(w) for w in words]

      vocab_list = list(SPECIALS)
      vocab_seen = set(vocab_list)

      initial_symbols = {chr(b) for b in range(256)} | {WORD_END}

      for s in sorted(initial_symbols):
          if s not in vocab_seen:
              vocab_list.append(s)
              vocab_seen.add(s)

      # this loop recounts every pair every merge, which is far too slow
      # for the legal corpus. Replace it with incremental pair counts + pair_to_words.
      pair_counts = Counter()
      pair_to_words = defaultdict(set)
      for wi, symbols in enumerate(word_symbols):
          f = freqs[wi]
          for i in range(len(symbols) - 1):
              p = (symbols[i], symbols[i + 1])
              pair_counts[p] += f
              pair_to_words[p].add(wi)

      heap = [(-c, p) for p, c in pair_counts.items()]
      heapq.heapify(heap)

      while len(vocab_list) < self.vocab_size:
          best_pair, best_count = None, 0
          while heap:
              negc, p = heapq.heappop(heap)
              if pair_counts.get(p, 0) == -negc and -negc > 0:
                  best_pair, best_count = p, -negc
                  break
          if best_pair is None or best_count <= 1:
              break

          a, b = best_pair
          merged = a + b
          changed = set()

          for wi in list(pair_to_words[best_pair]):
              symbols = word_symbols[wi]
              f = freqs[wi]

              old_pairs = Counter(zip(symbols, symbols[1:]))   # positions, so overlaps count
              new_symbols = []
              i = 0
              while i < len(symbols):
                  if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
                      new_symbols.append(merged)
                      i += 2
                  else:
                      new_symbols.append(symbols[i])
                      i += 1
              word_symbols[wi] = new_symbols
              new_pairs = Counter(zip(new_symbols, new_symbols[1:]))

              for p, c in old_pairs.items():
                  pair_counts[p] -= c * f
                  changed.add(p)
                  if p not in new_pairs:
                      pair_to_words[p].discard(wi)
              for p, c in new_pairs.items():
                  pair_counts[p] += c * f
                  changed.add(p)
                  pair_to_words[p].add(wi)

          for p in changed:
              c = pair_counts[p]
              if c > 0:
                  heapq.heappush(heap, (-c, p))
              else:
                  del pair_counts[p]
                  pair_to_words.pop(p, None)

          self.merges.append([a, b])
          self.merge_ranks[(a, b)] = len(self.merges) - 1

          if merged not in vocab_seen:
              vocab_list.append(merged)
              vocab_seen.add(merged)

      self.itos = vocab_list
      self.stoi = {tok: i for i, tok in enumerate(self.itos)}
      self.pad_id = self.stoi["<pad>"]
      self.bos_id = self.stoi["<s>"]
      self.eos_id = self.stoi["</s>"]
      return self

  def encode_word(self, word):
      if word in self._cache:
        return self._cache[word]
      symbols = self._word_to_symbols(word)
      while len(symbols) > 1:
          pairs = [(symbols[i], symbols[i + 1]) for i in range(len(symbols) - 1)]
          ranked = [(self.merge_ranks[p], i) for i, p in enumerate(pairs) if p in self.merge_ranks]
          if not ranked:
              break
          _, i = min(ranked)
          symbols = symbols[:i] + [symbols[i] + symbols[i + 1]] + symbols[i + 2:]

      self._cache[word] = symbols
      return symbols

  def encode_ids(self, sentence):
      unk = self.stoi["<unk>"]
      return [self.stoi.get(t, unk) for t in self.encode(sentence)]

  def decode(self, ids):
      text = "".join(self.itos[i] for i in ids if self.itos[i] not in SPECIALS)
      words = text.split(WORD_END)
      out = [bytes(ord(c) for c in w).decode("utf-8", errors="replace") for w in words if w]
      return " ".join(out)

  def encode(self, sentence):
      tokens = []
      for word in sentence.split():
          tokens.extend(self.encode_word(word))
      return tokens

  def save(self, path):
      with open(path, "w", encoding="utf-8") as f:
          json.dump({"merges": self.merges, "itos": self.itos,
                      "vocab_size": self.vocab_size}, f, ensure_ascii=False)

  @staticmethod
  def load_bpe(path):
      with open(path, "r", encoding="utf-8") as f:
          data = json.load(f)
      bpe = ByteLevelBPE(vocab_size=data["vocab_size"])
      bpe.merges = data["merges"]
      bpe.itos = data["itos"]
      bpe.stoi = {tok: i for i, tok in enumerate(bpe.itos)}
      bpe.merge_ranks = {(a, b): i for i, (a, b) in enumerate(bpe.merges)}
      bpe.pad_id = bpe.stoi["<pad>"]
      bpe.bos_id = bpe.stoi["<s>"]
      bpe.eos_id = bpe.stoi["</s>"]
      return bpe

In [ ]:
import hashlib
for k, f in PHASE1_BPE_FILES.items():   # all four must be the same tokenizer (needed for KD)
  print(k, hashlib.md5(open(f, "rb").read()).hexdigest())
bpe = ByteLevelBPE.load_bpe(PHASE1_BPE_FILES[MODEL_TYPE])
UNK_ID = bpe.stoi["<unk>"]
print("vocab size:", len(bpe.itos))

## 5. Download the Phase-4 data

Every file is read into a list of `{"id", "text", "label"}`. A list of objects (as in the brief) or a dictionary `{id: {...}}` is accepted; the label may be stored as `label`, `labels`, `decision` or `y`.

In [ ]:
from huggingface_hub import hf_hub_download
from collections import Counter

def load_json_file(fname):
  p = hf_hub_download(repo_id=HF_DATASET_REPO, filename=fname, repo_type="dataset", token=HF_TOKEN)
  with open(p, "r", encoding="utf-8") as f:
    return json.load(f)

def read_cases(data, with_labels):
  if isinstance(data, dict) and "data" in data and isinstance(data["data"], list):
    data = data["data"]
  if isinstance(data, dict):
    items = []
    for k, v in data.items():
      if isinstance(v, str):
        v = {"text": v}
      v = dict(v)
      v.setdefault("id", k)
      items.append(v)
    data = items
  cases = []
  for i, d in enumerate(data):
    text = d.get("text", "")
    if isinstance(text, list):
      text = " ".join(str(t) for t in text)
    label = None
    if with_labels:
      for key in ("label", "labels", "decision", "y"):
        if key in d:
          label = int(d[key])
          break
      if label is None:
        raise ValueError(f"no label found in example {d.get('id', i)}; keys: {list(d.keys())}")
    cases.append({"id": d.get("id", i), "text": str(text), "label": label})
  return cases

train_cases = read_cases(load_json_file(TRAIN_FILE), True)
dev_cases = read_cases(load_json_file(DEV_FILE), True)
test_cases = read_cases(load_json_file(TEST_FILE), False)
print(f"train: {len(train_cases)} cases | dev: {len(dev_cases)} | test: {len(test_cases)}")
for name, cs in [("train", train_cases), ("dev", dev_cases)]:
  c = Counter(x["label"] for x in cs)
  print(f"{name} labels (class balance): {dict(sorted(c.items()))}")
lens = sorted(len(x["text"].split()) for x in train_cases)
print(f"words per case: median {lens[len(lens) // 2]}, 90% {lens[int(0.9 * len(lens))]}, max {lens[-1]}")

if MAX_TRAIN_DOCS is not None and len(train_cases) > MAX_TRAIN_DOCS:
  rng = random.Random(SEED)
  train_cases = rng.sample(train_cases, MAX_TRAIN_DOCS)
  print(f"using a random subset of {len(train_cases)} training cases, labels {dict(sorted(Counter(x['label'] for x in train_cases).items()))}")

## 6. Long documents -> chunks

A judgment is far longer than one window, so every case becomes `N_CHUNKS` chunks of `CHUNK_TOKENS` tokens:
* `"last"`: the last `N_CHUNKS * CHUNK_TOKENS` tokens (the end of a judgment holds the court's reasoning),
* `"first"`: the beginning (facts, parties),
* `"first_last"`: half the chunks from the beginning and half from the end,
* `"spread"`: `N_CHUNKS` chunks spread evenly over the whole document.

For `"first"` and `"last"` only the needed words are tokenized (every word is at least one token), which saves a lot of time on long cases.

In [ ]:
from datasets import Dataset

def case_chunks(text):
  words = text.split()
  need = N_CHUNKS * CHUNK_TOKENS
  if SECTION == "last":
    ids = bpe.encode_ids(" ".join(words[-need:]))[-need:]
  elif SECTION == "first":
    ids = bpe.encode_ids(" ".join(words[:need]))[:need]
  else:
    ids = bpe.encode_ids(" ".join(words))
  if len(ids) == 0:
    ids = [bpe.eos_id]
  if SECTION in ("last", "first") or len(ids) <= need:
    pieces = [ids[i:i + CHUNK_TOKENS] for i in range(0, len(ids), CHUNK_TOKENS)]
  elif SECTION == "first_last":
    half = N_CHUNKS // 2
    head = ids[:half * CHUNK_TOKENS]
    tail = ids[-(N_CHUNKS - half) * CHUNK_TOKENS:]
    pieces = [head[i:i + CHUNK_TOKENS] for i in range(0, len(head), CHUNK_TOKENS)]
    pieces += [tail[i:i + CHUNK_TOKENS] for i in range(0, len(tail), CHUNK_TOKENS)]
  else:   # "spread": N_CHUNKS evenly spaced chunks over the whole document
    pieces = []
    n_possible = len(ids) - CHUNK_TOKENS
    for j in range(N_CHUNKS):
      st = round(j * n_possible / (N_CHUNKS - 1)) if N_CHUNKS > 1 else n_possible
      pieces.append(ids[st:st + CHUNK_TOKENS])
  pieces = pieces[:N_CHUNKS]
  out = []
  for p in pieces:
    out.append(p + [bpe.pad_id] * (CHUNK_TOKENS - len(p)))
  while len(out) < N_CHUNKS:                       # short case: empty chunks (all padding) are ignored by the model
    out.append([bpe.pad_id] * CHUNK_TOKENS)
  return out

t0 = time.time()
for x in train_cases + dev_cases + test_cases:
  x["chunks"] = case_chunks(x["text"])
  bpe._cache.clear() if len(bpe._cache) > 200000 else None
print(f"tokenized {len(train_cases) + len(dev_cases) + len(test_cases)} cases in {time.time() - t0:.0f}s")
real = [sum(1 for c in x["chunks"] if c[0] != bpe.pad_id) for x in train_cases]
print("real chunks per training case:", dict(sorted(Counter(real).items())))

train_dataset = Dataset.from_dict({"input_ids": [x["chunks"] for x in train_cases], "labels": [x["label"] for x in train_cases]})
val_dataset = Dataset.from_dict({"input_ids": [x["chunks"] for x in dev_cases], "labels": [x["label"] for x in dev_cases]})
print(train_dataset)
print(f"{len(train_cases) * N_CHUNKS} sequences of {CHUNK_TOKENS} tokens per epoch")

def data_collator(features):
  return {"input_ids": torch.tensor([f["input_ids"] for f in features], dtype=torch.long),
          "labels": torch.tensor([f["labels"] for f in features], dtype=torch.long)}

## 7. Phase-1 model code (copied unchanged from the Phase-1 notebook)

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class RMSNorm(nn.Module):
    """Module 2.3:  RMS(x) = sqrt( (1/E) * sum_i x_i^2 + eps ),   RMSNorm(x)_i = gamma_i * x_i / RMS(x)"""
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(d_model))   # learnable, no bias (beta)

    def forward(self, x):
        rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return self.gamma * (x / rms)


# ---- RoPE (Module 2.3 / 3.1, Assignment 3 "rope.py") ----
def build_rope_cache(max_seq_len, d_head, base=10000.0):
    # theta_i = base^(-2(i-1)/d) for every pair of dimensions; angle = position m * theta_i
    inv_freq = 1.0 / (base ** (torch.arange(0, d_head, 2).float() / d_head))   # (d_head/2,)
    positions = torch.arange(max_seq_len).float()
    freqs = torch.outer(positions, inv_freq)                                  # (T, d_head/2)
    angles = torch.cat([freqs, freqs], dim=-1)                                # (T, d_head)
    return angles.cos(), angles.sin()


def rotate_half(x):
    # split the last dim in half and return (-x2, x1)
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2:]
    return torch.cat([-x2, x1], dim=-1)


def apply_rope(x, cos, sin):
    # x: (B, n_heads, T, d_head); cos, sin: (B, 1, T, d_head)
    return x * cos + rotate_half(x) * sin


class CausalSelfAttention(nn.Module):
    """Masked multi-head self-attention (Module 2.3, Assignment 3).
    n_kv_heads == n_heads -> normal MHA,  n_kv_heads < n_heads -> grouped-query attention (GQA).
    past_kv = (k, v) of earlier tokens -> KV cache for generation (Module 2.3).
    attention_mask (B, S): 1 = real token, 0 = padding (Module 2.3: masks prevent attention over padding)."""
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.1, use_rope=False):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.use_rope = use_rope
        self.attention_scale = 1.0 / math.sqrt(self.d_head)

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", mask, persistent=False)
        if use_rope:
            cos, sin = build_rope_cache(max_seq_len, self.d_head)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, x, position_ids, attention_mask=None, past_kv=None):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)

        if self.use_rope:
            cos = self.rope_cos[position_ids].unsqueeze(1)   # (B, 1, T, d_head)
            sin = self.rope_sin[position_ids].unsqueeze(1)
            q = apply_rope(q, cos, sin)
            k = apply_rope(k, cos, sin)

        if past_kv is not None:                  # KV cache: reuse keys/values of earlier tokens
            k = torch.cat([past_kv[0], k], dim=2)
            v = torch.cat([past_kv[1], v], dim=2)
        new_kv = (k, v)

        if self.n_rep > 1:                       # GQA: each k/v head is shared by n_rep query heads
            k = torch.repeat_interleave(k, self.n_rep, dim=1)
            v = torch.repeat_interleave(v, self.n_rep, dim=1)

        S = k.size(2)                             # number of keys = cached + new
        scores = (q @ k.transpose(-2, -1)) * self.attention_scale        # (B, H, T, S)
        allowed = self.causal_mask[S - T:S, :S].unsqueeze(0).unsqueeze(0)  # (1, 1, T, S)
        if attention_mask is not None:
            allowed = allowed & attention_mask[:, None, None, :].bool()
            # a left-padding position has no real token before it, so let every
            # position always see itself (otherwise its whole row would be masked)
            itself = torch.eye(S, dtype=torch.bool, device=x.device)[S - T:S, :S]
            allowed = allowed | itself.unsqueeze(0).unsqueeze(0)
        scores = scores.masked_fill(~allowed, float("-inf"))
        att = F.softmax(scores, dim=-1)
        att = self.attn_dropout(att)
        out = att @ v                                                      # (B, H, T, d_head)
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.resid_dropout(self.out_proj(out))
        return out, new_kv


class FeedForward(nn.Module):
    """Assignment 3: two linear layers with a GELU in between, plus dropout."""
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model), nn.Dropout(dropout))

    def forward(self, x):
        return self.net(x)


class SwiGLUFeedForward(nn.Module):
    """Module 3.1 names SwiGLU as the modern feed-forward activation:
    FFN(x) = W2( SiLU(W1 x) * (W3 x) ),  SiLU(a) = a * sigmoid(a)"""
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff)
        self.w3 = nn.Linear(d_model, d_ff)
        self.w2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        a = self.w1(x)
        return self.dropout(self.w2(a * torch.sigmoid(a) * self.w3(x)))

In [ ]:
from dataclasses import dataclass


@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 640
    n_layers: int = 10
    n_heads: int = 10
    n_kv_heads: int = 10            # == n_heads for plain MHA
    d_ff: int = 2560
    max_seq_len: int = 2048
    dropout: float = 0.1
    pos_emb_type: str = "learned"   # "learned" (GPT-2 / BERT, Module 2.3) or "rope"
    norm_type: str = "layernorm"    # "layernorm" or "rmsnorm"
    ffn_type: str = "gelu"          # "gelu" or "swiglu"


class Block(nn.Module):
    """Pre-norm block (Module 2.3, used by GPT-2 and modern LLMs):  x + Attn(Norm(x)),  then  x + FFN(Norm(x))"""
    def __init__(self, cfg):
        super().__init__()
        if cfg.norm_type == "layernorm":
            self.ln1 = nn.LayerNorm(cfg.d_model)
            self.ln2 = nn.LayerNorm(cfg.d_model)
        else:
            self.ln1 = RMSNorm(cfg.d_model)
            self.ln2 = RMSNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.max_seq_len,
                                        cfg.dropout, use_rope=(cfg.pos_emb_type == "rope"))
        if cfg.ffn_type == "gelu":
            self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)
        else:
            self.ffn = SwiGLUFeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x, position_ids, attention_mask=None, past_kv=None):
        a, new_kv = self.attn(self.ln1(x), position_ids, attention_mask, past_kv)
        x = x + a
        x = x + self.ffn(self.ln2(x))
        return x, new_kv


class DecoderLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        if cfg.pos_emb_type == "learned":
            self.pos_emb = nn.Embedding(cfg.max_seq_len, cfg.d_model)    # learned absolute positions
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        if cfg.norm_type == "layernorm":
            self.ln_f = nn.LayerNorm(cfg.d_model)
        else:
            self.ln_f = RMSNorm(cfg.d_model)
        # Module 2.1 "Proper Weight Initialization": Xavier/Glorot for the embedding matrices.
        # PyTorch's default N(0, 1) embedding + weight tying gives huge first logits (start loss ~80)
        nn.init.xavier_uniform_(self.tok_emb.weight)
        if cfg.pos_emb_type == "learned":
            nn.init.xavier_uniform_(self.pos_emb.weight)

    def forward(self, input_ids, labels=None, attention_mask=None, position_ids=None,
                past_key_values=None, use_cache=False):
        B, T = input_ids.shape
        past_len = 0 if past_key_values is None else past_key_values[0][0].size(2)
        if position_ids is None:
            position_ids = torch.arange(past_len, past_len + T, device=input_ids.device).unsqueeze(0).expand(B, T)

        x = self.tok_emb(input_ids)
        if self.cfg.pos_emb_type == "learned":
            x = x + self.pos_emb(position_ids)
        x = self.drop(x)

        new_past = []
        for i, block in enumerate(self.blocks):
            layer_past = None if past_key_values is None else past_key_values[i]
            x, kv = block(x, position_ids, attention_mask, layer_past)
            new_past.append(kv)
        x = self.ln_f(x)

        if use_cache:
            x = x[:, -1:, :]              # generation only needs the prediction for the last position
        logits = x @ self.tok_emb.weight.t()   # weight tying

        loss = None
        if labels is not None:
            # predict token t+1 from positions <= t (labels are shifted inside, like GPT-2)
            loss = F.cross_entropy(logits[:, :-1, :].reshape(-1, logits.size(-1)).float(),
                                   labels[:, 1:].reshape(-1))
        if use_cache:
            return {"loss": loss, "logits": logits, "past_key_values": new_past}
        return {"loss": loss, "logits": logits}

In [ ]:
class LSTMCell(nn.Module):
    """One time step of an LSTM (Assignment 2): packed linear layers compute f, i, g, o at once."""
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.x2g = nn.Linear(input_size, 4 * hidden_size)
        self.h2g = nn.Linear(hidden_size, 4 * hidden_size)
        with torch.no_grad():
            self.x2g.bias[0:hidden_size] = 1.0   # forget gate bias close to 1 (Module 2.1)
            self.h2g.bias[0:hidden_size] = 0.0

    def forward(self, x_t, state):
        h_prev, c_prev = state
        gates = self.x2g(x_t) + self.h2g(h_prev)
        f_pre, i_pre, g_pre, o_pre = gates.chunk(4, dim=-1)
        f_t = torch.sigmoid(f_pre)
        i_t = torch.sigmoid(i_pre)
        g_t = torch.tanh(g_pre)
        o_t = torch.sigmoid(o_pre)
        c_t = f_t * c_prev + i_t * g_t
        h_t = o_t * torch.tanh(c_t)
        return (h_t, c_t)


class GRUCell(nn.Module):
    """One time step of a GRU with the Module 2.1 equations (layer names as in Assignment 2)."""
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.x2z = nn.Linear(input_size, hidden_size)
        self.h2z = nn.Linear(hidden_size, hidden_size, bias=False)
        self.x2r = nn.Linear(input_size, hidden_size)
        self.h2r = nn.Linear(hidden_size, hidden_size, bias=False)
        self.x2n = nn.Linear(input_size, hidden_size)
        self.h2n = nn.Linear(hidden_size, hidden_size, bias=False)

    def forward(self, x_t, h_prev):
        z_t = torch.sigmoid(self.x2z(x_t) + self.h2z(h_prev))          # update gate
        r_t = torch.sigmoid(self.x2r(x_t) + self.h2r(h_prev))          # reset gate
        n_t = torch.tanh(self.x2n(x_t) + self.h2n(r_t * h_prev))       # candidate h~
        h_t = (1 - z_t) * h_prev + z_t * n_t
        return h_t


class RNNLM(nn.Module):
    def __init__(self, vocab_size, hidden_size, num_layers, cell_type="lstm", dropout=0.1):
        super().__init__()
        self.is_lstm = (cell_type == "lstm")
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.tok_emb = nn.Embedding(vocab_size, hidden_size)
        cell_cls = LSTMCell if self.is_lstm else GRUCell
        self.cells = nn.ModuleList([cell_cls(hidden_size, hidden_size) for _ in range(num_layers)])
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.tok_emb.weight)     # Module 2.1: Xavier/Glorot initialisation

    def forward(self, input_ids, labels=None, attention_mask=None, past_key_values=None, use_cache=False):
        """past_key_values here = list of per-layer states (h, c) / h, carried during generation.
        attention_mask (B, T): at padding positions the state is NOT updated."""
        B, T = input_ids.shape
        x = self.dropout(self.tok_emb(input_ids))
        new_states = []
        for l in range(self.num_layers):
            cell = self.cells[l]
            if past_key_values is not None:
                state = past_key_values[l]
            else:
                h = torch.zeros(B, self.hidden_size, device=x.device, dtype=x.dtype)
                state = (h, torch.zeros_like(h)) if self.is_lstm else h
            outputs = []
            for t in range(T):           # explicit unroll over time (BPTT through these steps)
                new_state = cell(x[:, t, :], state)
                if attention_mask is not None:
                    m = attention_mask[:, t].unsqueeze(1).to(x.dtype)     # 1 = real token, 0 = pad
                    if self.is_lstm:
                        new_state = (m * new_state[0] + (1 - m) * state[0], m * new_state[1] + (1 - m) * state[1])
                    else:
                        new_state = m * new_state + (1 - m) * state
                state = new_state
                outputs.append(state[0] if self.is_lstm else state)
            new_states.append(state)
            x = self.dropout(torch.stack(outputs, dim=1))    # input of the next layer

        if use_cache:
            x = x[:, -1:, :]
        logits = x @ self.tok_emb.weight.t()       # weight tying (Assignment 3)

        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits[:, :-1, :].reshape(-1, logits.size(-1)).float(),
                                   labels[:, 1:].reshape(-1))
        if use_cache:
            return {"loss": loss, "logits": logits, "past_key_values": new_states}
        return {"loss": loss, "logits": logits}

In [ ]:
class T5RelativePositionBias(nn.Module):
    def __init__(self, n_heads, bidirectional, num_buckets=32, max_distance=128):
        super().__init__()
        self.bidirectional = bidirectional
        self.num_buckets = num_buckets
        self.max_distance = max_distance
        self.rel_emb = nn.Embedding(num_buckets, n_heads)     # one scalar per (bucket, head)

    def bucket(self, relative_position):
        # relative_position = key_pos - query_pos
        n = -relative_position
        num_buckets = self.num_buckets
        ret = torch.zeros_like(n)
        if self.bidirectional:                 # encoder: half the buckets for keys on the right
            num_buckets = num_buckets // 2
            ret = ret + (n < 0).long() * num_buckets
            n = torch.abs(n)
        else:                                  # decoder: only keys on the left exist
            n = torch.clamp(n, min=0)
        max_exact = num_buckets // 2           # small offsets get their own bucket
        is_small = n < max_exact
        # bigger offsets: logarithmically wider buckets, everything beyond max_distance shares the last one
        large = max_exact + (torch.log(torch.clamp(n, min=1).float() / max_exact)
                             / math.log(self.max_distance / max_exact)
                             * (num_buckets - max_exact)).long()
        large = torch.clamp(large, max=num_buckets - 1)
        return ret + torch.where(is_small, n, large)

    def forward(self, q_len, k_len, device):
        q_pos = torch.arange(q_len, device=device)[:, None]
        k_pos = torch.arange(k_len, device=device)[None, :]
        buckets = self.bucket(k_pos - q_pos)               # (q, k)
        bias = self.rel_emb(buckets)                       # (q, k, H)
        return bias.permute(2, 0, 1).unsqueeze(0)          # (1, H, q, k)


class T5Attention(nn.Module):
    """Self-attention (x_kv = x_q) or cross-attention (x_kv = encoder output)."""
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.attention_scale = 1.0 / math.sqrt(self.d_head)
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)

    def forward(self, x_q, x_kv, position_bias=None, allowed=None):
        B, Tq, C = x_q.shape
        Tk = x_kv.size(1)
        q = self.q_proj(x_q).view(B, Tq, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x_kv).view(B, Tk, self.n_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x_kv).view(B, Tk, self.n_heads, self.d_head).transpose(1, 2)
        scores = (q @ k.transpose(-2, -1)) * self.attention_scale
        if position_bias is not None:
            scores = scores + position_bias
        if allowed is not None:
            scores = scores.masked_fill(~allowed, float("-inf"))
        att = self.attn_dropout(F.softmax(scores, dim=-1))
        out = (att @ v).transpose(1, 2).contiguous().view(B, Tq, C)
        return self.out_proj(out)


class T5FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.fc2(self.dropout(F.relu(self.fc1(x))))   # dropout inside the FFN


class T5EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.self_attn = T5Attention(d_model, n_heads, dropout)
        self.norm2 = RMSNorm(d_model)
        self.ffn = T5FeedForward(d_model, d_ff, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, position_bias, src_allowed):
        h = self.norm1(x)
        x = x + self.dropout(self.self_attn(h, h, position_bias, src_allowed))
        x = x + self.dropout(self.ffn(self.norm2(x)))
        return x


class T5DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.self_attn = T5Attention(d_model, n_heads, dropout)
        self.norm2 = RMSNorm(d_model)
        self.cross_attn = T5Attention(d_model, n_heads, dropout)
        self.norm3 = RMSNorm(d_model)
        self.ffn = T5FeedForward(d_model, d_ff, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out, position_bias, causal_allowed, src_allowed):
        h = self.norm1(x)
        x = x + self.dropout(self.self_attn(h, h, position_bias, causal_allowed))   # masked self-attention
        h = self.norm2(x)
        x = x + self.dropout(self.cross_attn(h, enc_out, None, src_allowed))        # cross-attention
        x = x + self.dropout(self.ffn(self.norm3(x)))
        return x


class T5Model(nn.Module):
    def __init__(self, vocab_size, pad_id, d_model, n_enc_layers, n_dec_layers, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.pad_id = pad_id
        self.shared_emb = nn.Embedding(vocab_size, d_model)    # encoder input, decoder input and output
        self.dropout = nn.Dropout(dropout)
        self.enc_layers = nn.ModuleList([T5EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_enc_layers)])
        self.dec_layers = nn.ModuleList([T5DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_dec_layers)])
        self.enc_norm = RMSNorm(d_model)
        self.dec_norm = RMSNorm(d_model)
        self.enc_pos_bias = T5RelativePositionBias(n_heads, bidirectional=True)
        self.dec_pos_bias = T5RelativePositionBias(n_heads, bidirectional=False)
        nn.init.xavier_uniform_(self.shared_emb.weight)  # Module 2.1: Xavier/Glorot initialisation

    def encode(self, input_ids):
        S = input_ids.size(1)
        src_allowed = (input_ids != self.pad_id)[:, None, None, :]          # no attention to padding
        position_bias = self.enc_pos_bias(S, S, input_ids.device)
        x = self.dropout(self.shared_emb(input_ids))
        for layer in self.enc_layers:
            x = layer(x, position_bias, src_allowed)
        return self.dropout(self.enc_norm(x))

    def decode(self, decoder_input_ids, enc_out, input_ids, last_only=False):
        T = decoder_input_ids.size(1)
        src_allowed = (input_ids != self.pad_id)[:, None, None, :]
        causal_allowed = torch.tril(torch.ones(T, T, dtype=torch.bool, device=enc_out.device))[None, None]
        position_bias = self.dec_pos_bias(T, T, enc_out.device)
        x = self.dropout(self.shared_emb(decoder_input_ids))
        for layer in self.dec_layers:
            x = layer(x, enc_out, position_bias, causal_allowed, src_allowed)
        x = self.dropout(self.dec_norm(x))
        if last_only:
            x = x[:, -1:, :]
        return x @ self.shared_emb.weight.t()                                # weight sharing

    def forward(self, input_ids, decoder_input_ids, labels=None):
        enc_out = self.encode(input_ids)
        logits = self.decode(decoder_input_ids, enc_out, input_ids)
        loss = None
        if labels is not None:
            # decoder_input_ids are already shifted (<s> + target[:-1]); -100 = padding, ignored
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)).float(), labels.reshape(-1), ignore_index=-100)
        return {"loss": loss, "logits": logits}

### 7.5 Load a Phase-1 model, add the document classifier and the adaptation method

* **Chunk vector:** the Phase-1 model reads every chunk (for `t5` only the bidirectional encoder); the chunk vector is the mean of its token states. Empty (all-padding) chunks are skipped.
* **Document vector (`DOC_POOL`):** `"mean"` of the chunk vectors; `"attention"`: a learned score per chunk, `a = softmax(w . tanh(W v_j))`, document = `sum_j a_j v_j` (attention as weighted averaging, Module 2.3); `"bigru"`: the Phase-1 `GRUCell` reads the chunks forwards and backwards (Module 2.1) and the two final states are concatenated.
* **Head:** dropout + linear -> 2 classes, cross entropy with class weights `N / (2 * count)`.
* **Full / partial** fine-tuning as before. **BitFit** [8]: only the bias terms of the Phase-1 model (and the new head) are trained. **LoRMA** [12] as in Phases 2-3. **Adapters** (Houlsby et al. 2019; one adapter after every feed-forward layer as in Pfeiffer et al. 2020): `h + W_up ReLU(W_down h)` with a `BOTTLENECK`-size layer, `W_up` starts at zero so training starts from the Phase-1 model.
* **KD** [2]: as in Phases 2-3, on the 2-class distributions.

In [ ]:
import re
from safetensors.torch import load_file

def build_base(model_type, n_layers=None):
  V = len(bpe.itos)
  if model_type == "gpt":
    return DecoderLM(LMConfig(vocab_size=V, d_model=N_EMBD, n_layers=n_layers or N_LAYER, n_heads=N_HEAD,
                              n_kv_heads=N_HEAD, d_ff=D_FF, max_seq_len=MODEL_CONTEXT_LENGTH, dropout=DROPOUT,
                              pos_emb_type="learned", norm_type="layernorm", ffn_type="gelu"))
  if model_type == "modern":
    return DecoderLM(LMConfig(vocab_size=V, d_model=N_EMBD, n_layers=n_layers or N_LAYER, n_heads=N_HEAD,
                              n_kv_heads=N_KV_HEAD, d_ff=D_FF, max_seq_len=MODEL_CONTEXT_LENGTH, dropout=DROPOUT,
                              pos_emb_type="rope", norm_type="rmsnorm", ffn_type="swiglu"))
  if model_type == "rnn":
    return RNNLM(V, RNN_HIDDEN, n_layers or RNN_LAYERS, cell_type=RNN_CELL, dropout=DROPOUT)
  if model_type == "t5":
    return T5Model(V + T5_N_SENTINELS, bpe.pad_id, T5_D_MODEL, n_layers or T5_ENC_LAYERS, T5_DEC_LAYERS,
                   T5_HEADS, T5_D_FF, dropout=DROPOUT)
  raise ValueError(model_type)

def hidden_size(model_type):
  return {"gpt": N_EMBD, "modern": N_EMBD, "rnn": RNN_HIDDEN, "t5": T5_D_MODEL}[model_type]

def load_phase1_weights(base, model_type, n_layers=None):
  path = PHASE1_MODEL_FILES[model_type]
  state = load_file(path)
  if n_layers is not None:          # student: keep only the first n_layers layers
    kept = {}
    for k, v in state.items():
      m = re.match(r"^(blocks|cells|enc_layers)\.(\d+)\.", k)
      if m and int(m.group(2)) >= n_layers:
        continue
      kept[k] = v
    state = kept
  missing, unexpected = base.load_state_dict(state, strict=False)
  print(f"loaded Phase-1 {model_type} weights from {path} | missing {len(missing)} | unexpected {len(unexpected)}")
  if missing:
    raise ValueError(f"Phase-1 weights do not match the architecture, missing: {missing[:5]}")

def lm_hidden(base, input_ids):
  # DecoderLM.forward without the LM output layer
  B, T = input_ids.shape
  pos = torch.arange(T, device=input_ids.device).unsqueeze(0).expand(B, T)
  x = base.tok_emb(input_ids)
  if base.cfg.pos_emb_type == "learned":
    x = x + base.pos_emb(pos)
  x = base.drop(x)
  for block in base.blocks:
    x, _ = block(x, pos)
  return base.ln_f(x)

def rnn_hidden(base, input_ids):
  # RNNLM.forward without the LM output layer
  B, T = input_ids.shape
  x = base.dropout(base.tok_emb(input_ids))
  for l in range(base.num_layers):
    cell = base.cells[l]
    h = torch.zeros(B, base.hidden_size, device=x.device, dtype=x.dtype)
    state = (h, torch.zeros_like(h)) if base.is_lstm else h
    outputs = []
    for t in range(T):
      state = cell(x[:, t, :], state)
      outputs.append(state[0] if base.is_lstm else state)
    x = base.dropout(torch.stack(outputs, dim=1))
  return x


class LoRALinear(nn.Module):
  """LoRA: h = W x + b + scale * B A x  (W, b frozen); scale = alpha / r (LoRA) or alpha / sqrt(r) (rsLoRA)"""
  def __init__(self, base_linear, r, alpha, rank_stabilized=False):
    super().__init__()
    self.base = base_linear
    self.scale = alpha / math.sqrt(r) if rank_stabilized else alpha / r
    self.A = nn.Parameter(torch.randn(r, base_linear.in_features) / math.sqrt(base_linear.in_features))
    self.B = nn.Parameter(torch.zeros(base_linear.out_features, r))

  def forward(self, x):
    return self.base(x) + self.scale * ((x @ self.A.t()) @ self.B.t())


class RSLoRALinear(LoRALinear):
  def __init__(self, base_linear, r, alpha):
    super().__init__(base_linear, r, alpha, rank_stabilized=True)


class LoRMALinear(nn.Module):
  """LoRMA (additive identity): h = (I + (alpha / r) * B A) W x + b  (W, b frozen)"""
  def __init__(self, base_linear, r, alpha):
    super().__init__()
    self.base = base_linear
    self.scale = alpha / r
    self.A = nn.Parameter(torch.randn(r, base_linear.out_features) / math.sqrt(base_linear.out_features))
    self.B = nn.Parameter(torch.zeros(base_linear.out_features, r))

  def forward(self, x):
    y = F.linear(x, self.base.weight)                      # W x
    y = y + self.scale * ((y @ self.A.t()) @ self.B.t())   # (I + s B A) W x
    if self.base.bias is not None:
      y = y + self.base.bias
    return y



def add_adapters(module, adapter_cls, r, alpha):
  # replace every nn.Linear inside the Phase-1 model by an adapter around it
  for name, child in module.named_children():
    if isinstance(child, nn.Linear):
      setattr(module, name, adapter_cls(child, r, alpha))
    else:
      add_adapters(child, adapter_cls, r, alpha)


class AdapterFFN(nn.Module):
  """Bottleneck adapter after a feed-forward layer: h = FFN(x);  h + W_up ReLU(W_down h).  W_up = 0 at the start."""
  def __init__(self, ffn, d_model, bottleneck):
    super().__init__()
    self.ffn = ffn
    self.down = nn.Linear(d_model, bottleneck)
    self.up = nn.Linear(bottleneck, d_model)
    nn.init.zeros_(self.up.weight)
    nn.init.zeros_(self.up.bias)

  def forward(self, x):
    h = self.ffn(x)
    return h + self.up(F.relu(self.down(h)))


def add_ffn_adapters(base, model_type, d_model):
  if model_type in ("gpt", "modern"):
    layers = base.blocks
  elif model_type == "t5":
    layers = base.enc_layers
  else:
    raise ValueError("adapters are only defined for the Transformer models (gpt, modern, t5)")
  for layer in layers:
    layer.ffn = AdapterFFN(layer.ffn, d_model, BOTTLENECK)


class CJPEModel(nn.Module):
  def __init__(self, base, model_type, d_model, dropout, class_weights):
    super().__init__()
    self.base = base
    self.model_type = model_type
    self.d_model = d_model
    self.dropout = nn.Dropout(dropout)
    if DOC_POOL == "attention":
      self.att_proj = nn.Linear(d_model, d_model)
      self.att_score = nn.Linear(d_model, 1)
      head_in = d_model
    elif DOC_POOL == "bigru":
      self.gru_fw = GRUCell(d_model, DOC_GRU_HIDDEN)
      self.gru_bw = GRUCell(d_model, DOC_GRU_HIDDEN)
      head_in = 2 * DOC_GRU_HIDDEN
    else:
      head_in = d_model
    self.head = nn.Linear(head_in, 2)
    self.register_buffer("class_weights", class_weights, persistent=False)
    self.last_attention = None

  def hidden(self, input_ids):
    if self.model_type == "t5":
      return self.base.encode(input_ids)                  # bidirectional encoder, padding masked
    if self.model_type == "rnn":
      return rnn_hidden(self.base, input_ids)
    return lm_hidden(self.base, input_ids)                # causal: right padding never affects real tokens

  def chunk_vectors(self, input_ids):
    B, N, T = input_ids.shape
    flat = input_ids.view(B * N, T)
    real = (flat != bpe.pad_id).any(dim=-1)              # which chunks contain text
    h = self.hidden(flat[real])                           # (R, T, D) only for real chunks
    m = (flat[real] != bpe.pad_id).unsqueeze(-1).to(h.dtype)
    v = (h * m).sum(dim=1) / m.sum(dim=1).clamp(min=1)    # mean over the real tokens of each chunk
    out = v.new_zeros(B * N, v.size(-1))
    out[real] = v
    return out.view(B, N, -1), real.view(B, N)

  def document_vector(self, v, present):
    p = present.to(v.dtype).unsqueeze(-1)                 # (B, N, 1)
    if DOC_POOL == "mean":
      return (v * p).sum(dim=1) / p.sum(dim=1).clamp(min=1)
    if DOC_POOL == "attention":
      scores = self.att_score(torch.tanh(self.att_proj(v))).squeeze(-1).float()   # (B, N)
      scores = scores.masked_fill(~present, -1e4)
      a = torch.softmax(scores, dim=-1).to(v.dtype)
      self.last_attention = a.detach()
      return (a.unsqueeze(-1) * v).sum(dim=1)
    # "bigru"
    B, N, D = v.shape
    h = v.new_zeros(B, DOC_GRU_HIDDEN)
    for j in range(N):
      h = p[:, j] * self.gru_fw(v[:, j], h) + (1 - p[:, j]) * h
    h_fw = h
    h = v.new_zeros(B, DOC_GRU_HIDDEN)
    for j in range(N - 1, -1, -1):
      h = p[:, j] * self.gru_bw(v[:, j], h) + (1 - p[:, j]) * h
    return torch.cat([h_fw, h], dim=-1)

  def forward(self, input_ids, labels=None):
    v, present = self.chunk_vectors(input_ids)
    d = self.document_vector(self.dropout(v), present)
    logits = self.head(self.dropout(d))
    loss = None
    if labels is not None:
      loss = F.cross_entropy(logits.float(), labels, weight=self.class_weights)
    return {"loss": loss, "logits": logits}


def set_trainable(model, ft_method):
  for p in model.parameters():
    p.requires_grad = (ft_method == "full")
  for name, p in model.named_parameters():
    if not name.startswith("base."):                 # document pooling + head are new: always trained
      p.requires_grad = True
  for name, p in model.base.named_parameters():
    if ft_method == "lorma" and (name.endswith(".A") or name.endswith(".B")):
      p.requires_grad = True
    if ft_method == "adapter" and (".down." in name or ".up." in name):
      p.requires_grad = True
    if ft_method == "bitfit" and name.endswith(".bias"):
      p.requires_grad = True
  if ft_method == "partial":
    if model.model_type in ("gpt", "modern"):
      layers, final_norm = model.base.blocks, model.base.ln_f
    elif model.model_type == "rnn":
      layers, final_norm = model.base.cells, None
    else:
      layers, final_norm = model.base.enc_layers, model.base.enc_norm
    for layer in list(layers)[-PARTIAL_TOP_LAYERS:]:
      for p in layer.parameters():
        p.requires_grad = True
    if final_norm is not None:
      for p in final_norm.parameters():
        p.requires_grad = True


def class_weights():
  w = torch.ones(2)
  if CLASS_WEIGHTING == "inverse":
    c = Counter(x["label"] for x in train_cases)
    for k in (0, 1):
      w[k] = len(train_cases) / (2 * max(c.get(k, 0), 1))
  return w


def build_cjpe_model(model_type, ft_method, n_layers=None, load_pretrained=True):
  base = build_base(model_type, n_layers)
  if load_pretrained:
    load_phase1_weights(base, model_type, n_layers)
  if model_type == "t5":
    base.dec_layers = nn.ModuleList()       # only the T5 encoder is used
  if ft_method == "lorma":
    add_adapters(base, LoRMALinear, ADAPTER_RANK, ADAPTER_ALPHA)
  elif ft_method == "adapter":
    add_ffn_adapters(base, model_type, hidden_size(model_type))
  model = CJPEModel(base, model_type, hidden_size(model_type), DROPOUT, class_weights())
  set_trainable(model, ft_method)
  return model


class KDModel(nn.Module):
  def __init__(self, student, teacher, temperature, alpha):
    super().__init__()
    self.student = student
    self.teacher = teacher
    self.T = temperature
    self.alpha = alpha
    for p in self.teacher.parameters():
      p.requires_grad = False

  def train(self, mode=True):
    super().train(mode)
    self.teacher.eval()                     # the teacher never uses dropout
    return self

  def forward(self, input_ids, labels=None):
    out = self.student(input_ids, labels)
    if labels is None:
      return out
    with torch.no_grad():
      t_logits = self.teacher(input_ids)["logits"]
    s = out["logits"].float()
    t = t_logits.float()
    kd = F.kl_div(F.log_softmax(s / self.T, dim=-1), F.softmax(t / self.T, dim=-1),
                  reduction="batchmean") * self.T * self.T
    return {"loss": self.alpha * out["loss"] + (1 - self.alpha) * kd, "logits": out["logits"]}


print("class weights:", class_weights().tolist())
if USE_KD:
  student = build_cjpe_model(MODEL_TYPE, FT_METHOD, n_layers=STUDENT_LAYERS)
  teacher = build_cjpe_model(TEACHER_MODEL_TYPE, TEACHER_FT_METHOD, load_pretrained=False)
  teacher.load_state_dict(torch.load(TEACHER_CJPE_PATH, map_location="cpu"))
  print("teacher loaded from", TEACHER_CJPE_PATH)
  model = KDModel(student, teacher, KD_TEMPERATURE, KD_ALPHA)
  cjpe_model = student
  n_teacher = sum(p.numel() for p in teacher.parameters())
else:
  cjpe_model = build_cjpe_model(MODEL_TYPE, FT_METHOD, load_pretrained=USE_PHASE1_WEIGHTS)
  model = cjpe_model
  n_teacher = 0

n_total = sum(p.numel() for p in cjpe_model.parameters())
n_trainable = sum(p.numel() for p in cjpe_model.parameters() if p.requires_grad)
print(f"CJPE model: {n_total / 1e6:.2f}M parameters | trainable {n_trainable / 1e6:.3f}M ({100 * n_trainable / n_total:.2f}%)")
if USE_KD:
  print(f"teacher: {n_teacher / 1e6:.2f}M parameters -> student is {100 * n_total / n_teacher:.0f}% of the teacher")

## 8. Training (Trainer, like the earlier phases)

The checkpoint is chosen by dev macro-F1.

In [ ]:
from transformers import TrainingArguments, Trainer

def binary_scores(gold, pred):
  per = {}
  for c in (0, 1):
    tp = sum(1 for g, p in zip(gold, pred) if g == c and p == c)
    fp = sum(1 for g, p in zip(gold, pred) if g != c and p == c)
    fn = sum(1 for g, p in zip(gold, pred) if g == c and p != c)
    P = tp / (tp + fp) if tp + fp else 0.0
    R = tp / (tp + fn) if tp + fn else 0.0
    per[c] = (P, R, 2 * P * R / (P + R) if P + R else 0.0)
  mp = (per[0][0] + per[1][0]) / 2
  mr = (per[0][1] + per[1][1]) / 2
  mf = (per[0][2] + per[1][2]) / 2
  acc = sum(1 for g, p in zip(gold, pred) if g == p) / max(len(gold), 1)
  return mp, mr, mf, acc, per

def compute_metrics(eval_pred):
  logits = eval_pred.predictions[0] if isinstance(eval_pred.predictions, tuple) else eval_pred.predictions
  pred = logits.argmax(axis=-1).tolist()
  return {"macro_f1": binary_scores(eval_pred.label_ids.tolist(), pred)[2]}

training_args = TrainingArguments(
  output_dir=os.path.join(EXP_DIR, "checkpoints"),
  num_train_epochs=EPOCHS,
  per_device_train_batch_size=BATCH_SIZE,
  per_device_eval_batch_size=BATCH_SIZE,
  gradient_accumulation_steps=GRAD_ACCUM,
  learning_rate=LEARNING_RATE,
  weight_decay=0.01,
  warmup_steps=5 if SMOKE_TEST else 50,
  lr_scheduler_type="cosine",
  max_grad_norm=1.0,
  fp16=torch.cuda.is_available() and MODEL_TYPE != "t5",   # T5 trains in full precision (fp16 broke it in Phase 2)
  eval_strategy="epoch",
  save_strategy="epoch",
  save_total_limit=2,
  load_best_model_at_end=True,          # checkpoint selection on dev macro-F1
  metric_for_best_model="macro_f1",
  greater_is_better=True,
  logging_steps=20,
  seed=SEED,
  report_to="none",
)

trainer = Trainer(model=model, args=training_args, train_dataset=train_dataset,
                  eval_dataset=val_dataset, data_collator=data_collator, compute_metrics=compute_metrics)

if torch.cuda.is_available():
  torch.cuda.reset_peak_memory_stats()
start_time = time.time()
if LOAD_CJPE_PATH:
  cjpe_model.load_state_dict(torch.load(LOAD_CJPE_PATH, map_location="cpu"))
  print("loaded trained CJPE model from", LOAD_CJPE_PATH, "- skipping training")
else:
  trainer.train()
training_time = time.time() - start_time
peak_memory_gb = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0
print(f"Training time: {training_time / 60:.1f} min | peak GPU memory: {peak_memory_gb:.2f} GB")
for h in trainer.state.log_history:
  if "eval_loss" in h:
    print(f"epoch {h['epoch']:.2f}: eval_loss {h['eval_loss']:.4f} | eval macro-F1 {h.get('eval_macro_f1', float('nan')):.4f}")
  elif "loss" in h:
    print(f"   step {h['step']}: train loss {h['loss']:.4f}")

## 9. Dev evaluation: macro-F1, threshold tuning and error analysis

* Scores at the default threshold 0.5 (argmax) and at the threshold on P(label 1) that maximises dev macro-F1 (`THRESHOLD_TUNING`).
* False positives / false negatives: their length and the end of a few of these judgments.
* Scores by document length (short / medium / long thirds).
* For `DOC_POOL = "attention"`: the average attention weight of every chunk position, i.e. which part of the judgment the model relies on.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cjpe_model = cjpe_model.to(device)
cjpe_model.eval()
PRED_BATCH = max(1, 16 // N_CHUNKS)

@torch.no_grad()
def predict_probs(cases):
  probs, atts = [], []
  for b in range(0, len(cases), PRED_BATCH):
    x = torch.tensor([c["chunks"] for c in cases[b:b + PRED_BATCH]], dtype=torch.long, device=device)
    if device.type == "cuda" and MODEL_TYPE != "t5":
      with torch.autocast(device_type="cuda", dtype=torch.float16):
        logits = cjpe_model(x)["logits"]
    else:
      logits = cjpe_model(x)["logits"]
    probs += torch.softmax(logits.float(), dim=-1)[:, 1].tolist()
    if cjpe_model.last_attention is not None:
      atts += cjpe_model.last_attention.float().tolist()
  return probs, atts

t0 = time.time()
dev_probs, dev_atts = predict_probs(dev_cases)
dev_seconds = time.time() - t0
dev_gold = [c["label"] for c in dev_cases]

def at_threshold(th):
  return [1 if p >= th else 0 for p in dev_probs]

mp, mr, mf_05, acc, per = binary_scores(dev_gold, at_threshold(0.5))
print(f"DEV threshold 0.50: macro P {mp:.4f} | R {mr:.4f} | F1 {mf_05:.4f} | accuracy {acc:.4f}   ({dev_seconds:.1f}s for {len(dev_cases)} cases)")
print(f"   class 0: P {per[0][0]:.3f} R {per[0][1]:.3f} F1 {per[0][2]:.3f} | class 1: P {per[1][0]:.3f} R {per[1][1]:.3f} F1 {per[1][2]:.3f}")

best_th, best_f1 = 0.5, mf_05
if THRESHOLD_TUNING:
  for i in range(5, 96):
    th = i / 100
    f = binary_scores(dev_gold, at_threshold(th))[2]
    if f > best_f1:
      best_th, best_f1 = th, f
THRESHOLD = best_th
dev_pred = at_threshold(THRESHOLD)
mp, mr, mf, acc, per = binary_scores(dev_gold, dev_pred)
print(f"DEV threshold {THRESHOLD:.2f} (tuned): macro P {mp:.4f} | R {mr:.4f} | F1 {mf:.4f} | accuracy {acc:.4f}")
maj = Counter(dev_gold).most_common(1)[0][0]
print(f"majority-label baseline macro-F1: {binary_scores(dev_gold, [maj] * len(dev_gold))[2]:.4f}")

# ---- error analysis ----
groups = {"TP": [], "TN": [], "FP": [], "FN": []}
for c, g, p in zip(dev_cases, dev_gold, dev_pred):
  key = ("T" if g == p else "F") + ("P" if p == 1 else "N")
  groups[key].append(c)
print("\nconfusion:", {k: len(v) for k, v in groups.items()})
for k, v in groups.items():
  if v:
    print(f"  {k}: mean length {sum(len(c['text'].split()) for c in v) / len(v):.0f} words")
for k in ("FP", "FN"):
  print(f"\nend of 3 {k} cases (gold {'0' if k == 'FP' else '1'}, predicted {'1' if k == 'FP' else '0'}):")
  for c in groups[k][:3]:
    print("   ...", c["text"][-300:].replace("\n", " "))

order = sorted(range(len(dev_cases)), key=lambda i: len(dev_cases[i]["text"].split()))
third = max(1, len(order) // 3)
for name, idx in [("short", order[:third]), ("medium", order[third:2 * third]), ("long", order[2 * third:])]:
  f = binary_scores([dev_gold[i] for i in idx], [dev_pred[i] for i in idx])[2]
  print(f"{name:6s} cases ({len(idx)}): macro-F1 {f:.4f}")

if dev_atts:
  avg = [sum(a[j] for a in dev_atts) / len(dev_atts) for j in range(N_CHUNKS)]
  print(f"\naverage attention weight per chunk position ({SECTION}, chunk 1 = earliest):", [round(x, 3) for x in avg])

## 10. Save the CJPE model (needed when this experiment is the KD teacher)

In [ ]:
torch.save(cjpe_model.state_dict(), os.path.join(EXP_DIR, "cjpe_model.pt"))
print("saved", os.path.join(EXP_DIR, "cjpe_model.pt"))

## 11. Test predictions -> prediction.json -> submission zip

A JSON list `[{"id": ..., "pred": 0 or 1}, ...]` with exactly one prediction per test case, using the dev-tuned threshold.

In [ ]:
t0 = time.time()
test_probs, _ = predict_probs(test_cases)
test_seconds = time.time() - t0
predictions = [{"id": c["id"], "pred": int(1 if p >= THRESHOLD else 0)} for c, p in zip(test_cases, test_probs)]
assert len(predictions) == len(test_cases) and len(set(str(p["id"]) for p in predictions)) == len(test_cases)
assert all(p["pred"] in (0, 1) for p in predictions)
with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
  json.dump(predictions, f)
with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as z:
  z.write(PREDICTION_PATH, arcname="prediction.json")
print(f"{len(predictions)} test predictions in {test_seconds:.0f}s | predicted labels:", dict(Counter(p["pred"] for p in predictions)))
print("submission:", SUBMISSION_ZIP)

## 12. Log this experiment (one line per experiment in results.jsonl)

In [ ]:
result = {
  "experiment": EXPERIMENT_NAME, "model": MODEL_TYPE, "method": FT_METHOD, "kd": USE_KD,
  "student_layers": STUDENT_LAYERS if USE_KD else None,
  "teacher": f"{TEACHER_MODEL_TYPE}/{TEACHER_FT_METHOD}" if USE_KD else None,
  "phase1_weights": USE_PHASE1_WEIGHTS, "train_docs": len(train_cases),
  "n_chunks": N_CHUNKS, "chunk_tokens": CHUNK_TOKENS, "section": SECTION, "doc_pool": DOC_POOL,
  "class_weighting": CLASS_WEIGHTING,
  "params_M": round(n_total / 1e6, 2), "trainable_M": round(n_trainable / 1e6, 3),
  "trainable_pct": round(100 * n_trainable / n_total, 2),
  "train_min": round(training_time / 60, 1), "peak_mem_GB": round(peak_memory_gb, 2),
  "test_pred_s": round(test_seconds, 1),
  "dev_F1_th05": round(mf_05, 4), "dev_F1_tuned": round(mf, 4), "threshold": THRESHOLD,
  "dev_P": round(mp, 4), "dev_R": round(mr, 4), "dev_acc": round(acc, 4),
  "lr": LEARNING_RATE, "epochs": EPOCHS,
}
with open(RESULTS_PATH, "a") as f:
  f.write(json.dumps(result) + "\n")
print(json.dumps(result, indent=1))